<a href="https://colab.research.google.com/github/SusanaBN30/Tareas/blob/main/Data_Analyst_Assignment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Data Analyst Assignment

## Data Challenge

Sales leadership has tasked you with analyzing the data to discover how employee training
impacts sales performance and effectiveness. The goal is to identify patterns and provide
actionable recommendations that drive sales growth and improve team performance by
answering the following questions.
Questions
1. What is the training completion rate for each course by segments (SVP Leader/Region),
factoring in the following caveats? Training is required for all employees except:
* Employees currently on leave are exempt from the training requirement.
* 'Sell More Suite SKU' course is not required for employees within the 'Advocacy'
cost center family.
* “Suite/Automation Technical Lab” and “Advanced Suite Bots Lab Course”
courses are required only for employees in the 'PreSales' and 'Services' cost
center family.

2. How would you analyze the performance of an account executive? Is there a difference
between those who have completed training and those who have not? How would you
segment the data and present your findings to senior stakeholders?
* Hint: Look at this from an overall employee perspective who has completed any
training vs. those who have not completed any training. Any training would count,

rather than distinguishing which specific courses were completed, as they all
contribute to one Suite product.

3. Any other interesting insights that you can see in the data? Any data quality issues with
the data? Any challenges with analyzing the data? What additional data do you think
would be useful for further analyzing the existing datasets?



---

## Data Dictionary

### Table 1 : Employee Information

This table maintains critical employment details for organization members, including
identification, job titles, management hierarchies, tenure, and work region

* **Employee_ID:** Unique identifier for each employee.
* **SVP Leader:** Leader overseeing the employee.
* **Business Title:** Official job title of the employee.
* **Cost Center:** Identifier for the employee's department or unit for cost tracking.
* **Cost Center Family:** Group of related cost centers for financial reporting.
* **Length of Service:** Total time the employee has worked at the company.
* **Leave Status:** Indication of whether the employee is on leave.
* **Is People Manager?:** Indicates if the employee oversees other staff.
* **Region:** Geographic area where the employee works.
* **Manager IC Helper:** Additional data supporting managerial status.
* **IC:** Individual Contributor

### Table 2: Completed Trainings
This table records the professional development activities of employees by linking completed
training programs to their unique identifiers
* **Employee_ID:** Unique identifier linked to an employee who completed the training.
* **Training Name:** Name of the training program or course that the employee completed.

### Table 3: Performance Data
This table includes the sales and revenue generation activities of employees by cataloguing
opportunities, their progression, and financial outcomes. It captures granular data on sales
stage milestones, product-related charges, and revenue figures, all linked by employee and
opportunity identifiers, making it a vital asset for analyzing sales performance and compensation
metrics.

* **Employee_ID:** Unique identifier for the employee associated.
* **Opportunity ID:** Unique identifier for the sales opportunity.
* **Type:** The category or classification of the opportunity.
* Expansion is for existing business

* **Stage 2+ Date:** The date when the opportunity reached or surpassed stage 2 in the
sales process.
* **Stage:** Current stage of the opportunity in the sales pipeline.
 * 02 - Discovery: Initial stage where potential needs and opportunities are
identified with the client.
 * 03 - Solution Review: Potential solutions are presented and reviewed with the
client.
 * 04 - Solution Validation: Client feedback is incorporated, and solutions are
refined and validated.
 * 05 - Contracting / Verba: Terms are negotiated and a verbal agreement may be
reached.
 * 06 - Signed/07 - Closed: Formal agreement is executed with signatures from all
parties. Signed and Closed are counted as finalized
* **Close Date:** The date when the opportunity was closed.
* **Product Rate Plan Charge:** The charge associated with the product's rate plan.
* **Product Name:** The name of the product related to the opportunity.
* **Add-On ARR (converted):** The value of the additional ARR from add-ons,
* **Total Commissionable ARR (converted):** The total annual recurring revenue that is
eligible for commission

In [17]:
import pandas as pd
import numpy as np
import plotly.express as px
archivo = 'Assignment_2.xlsx'
df_empleados = pd.read_excel(archivo, sheet_name='Employee_Data')
df_entrenamiento = pd.read_excel(archivo, sheet_name='Completed_Trainings')
df_ventas = pd.read_excel(archivo, sheet_name='Performance Data')

# Data Challenge


1.

In [6]:
empleados_obligados = df_empleados[df_empleados['Leave Status'] == 'Active'].copy()
def determinar_requerimiento(fila, curso):
    familia = fila['Cost Center Family']

    if curso == 'Sell More Suite SKU' and familia == 'Advocacy':
        return False

    cursos_tecnicos = ['Suite/Automation Technical Lab', 'Advanced Suite Bots Lab Course']
    if curso in cursos_tecnicos:
        return familia in ['PreSales', 'Services']

    return True

In [14]:
df_empleados.columns = df_empleados.columns.str.strip()
df_entrenamiento.columns = df_entrenamiento.columns.str.strip()

def obtener_tasa_correcta(df_emp, df_ent):
    activos = df_emp[df_emp['Leave Status'] == 'Active'].copy()
    cursos = df_ent['Training_Completed'].unique()
    resultados = []

    for curso in cursos:
        if curso == 'Sell More Suite SKU':
            obligados = activos[activos['Cost Center Family'] != 'Advocacy'].copy()
        elif curso in ['Suite/Automation Technical Lab', 'Advanced Suite Bots Lab Course']:
            obligados = activos[activos['Cost Center Family'].isin(['PreSales', 'Services'])].copy()
        else:
            obligados = activos.copy()

        terminaron = df_ent[df_ent['Training_Completed'] == curso]

        analisis_curso = pd.merge(obligados, terminaron, on='Employee_ID', how='left')

        analisis_curso['Finalizado'] = analisis_curso['Training_Completed'].notna().astype(int)

        resumen = analisis_curso.groupby(['SVP Leader', 'Region']).agg(
            total_obligados=('Employee_ID', 'count'),
            total_terminaron=('Finalizado', 'sum')
        ).reset_index()

        resumen['Curso'] = curso
        resumen['Tasa_Finalizacion_%'] = (resumen['total_terminaron'] / resumen['total_obligados']) * 100
        resultados.append(resumen)

    return pd.concat(resultados)

df_final_real = obtener_tasa_correcta(df_empleados, df_entrenamiento)
display(df_final_real.sort_values('Tasa_Finalizacion_%', ascending=False))

,SVP Leader,Region,total_obligados,total_terminaron,Curso,Tasa_Finalizacion_%
5,Leader 9,APAC,25,25,Advanced Suite Bots Lab Course,100.000000
8,Leader 9,North America,71,71,Suite/Automation Technical Lab,100.000000
5,Leader 9,APAC,25,25,Suite/Automation Technical Lab,100.000000
8,Leader 9,North America,71,71,Advanced Suite Bots Lab Course,100.000000
7,Leader 9,LATAM,16,16,Advanced Suite Bots Lab Course,100.000000
6,Leader 9,EMEA,45,45,Advanced Suite Bots Lab Course,100.000000
0,Leader 2,North America,1,1,Suite/Automation Technical Lab,100.000000
6,Leader 9,EMEA,45,45,Suite/Automation Technical Lab,100.000000
7,Leader 9,LATAM,16,16,Suite/Automation Technical Lab,100.000000
31,Leader 7,LATAM,1,1,Sell More Suite SKU,100.000000


2.

In [18]:
empleados_entrenados = df_entrenamiento['Employee_ID'].unique()
df_empleados['Estado_Entrenamiento'] = df_empleados['Employee_ID'].apply(
    lambda x: 'Trained' if x in empleados_entrenados else 'Untrained'
)
df_performance_final = pd.merge(df_ventas, df_empleados[['Employee_ID', 'Estado_Entrenamiento', 'Region']], on='Employee_ID', how='inner')


comparativa_ventas = df_performance_final.groupby('Estado_Entrenamiento').agg(
    promedio_ARR=('Total Commissionable ARR (converted)', 'mean'),
    oportunidades_totales=('Opportunity ID', 'count'),
    ventas_cerradas=('Stage', lambda x: x.isin(['06 - Signed', '07 - Closed']).sum())
).reset_index()


comparativa_ventas['Win_Rate_%'] = (comparativa_ventas['ventas_cerradas'] / comparativa_ventas['oportunidades_totales']) * 100

print("Desempeño de entrenados y no entrenados")
display(comparativa_ventas)

Desempeño de entrenados y no entrenados


,Estado_Entrenamiento,promedio_ARR,oportunidades_totales,ventas_cerradas,Win_Rate_%
0,Trained,107243.380531,339,46,13.569322
1,Untrained,77639.695489,266,36,13.533835


Notemos que el grupo entrenado genera más ingresos que el grupo no entrenado, con una gran diferencia, sin embargo, el número de tratos cerrados es muy similar, por lo que podemos decir que aunque no ayude a tener mas tratos cerrados, el entrenamiento si ayuda a que esos tratos generen mas ingresos.

3.

Podemos comparar el rendimiento de cada empleado antes del entrenamiento y despues del entrenamiento para confirmar que realmente hubo un mejor rendimiento gracias al entrenamiento recibido.